# Clustering de Clientes — Vértice Investimentos
**Aluno:** Gabriel Koji Kondo — 3°I  
**Disciplina:** Ciência de Dados  
**Dataset:** `carteiras_clientes.csv` — 500 clientes ativos

---

## 1. Entendimento do Problema

### O que o Rafael está pedindo?

O Head de Relacionamento da Vértice Investimentos percebeu que a equipe de relacionamento trata 500 clientes com a mesma abordagem — mesmo conteúdo educacional, mesmas ligações, mesmas sugestões de produto. O problema é que os clientes são completamente diferentes entre si.

Rafael quer saber: **quem são os clientes em termos de comportamento de investimento?** Não no sentido cadastral, mas nos padrões reais de como cada um investe.

### Hipótese analítica

> É possível identificar grupos de clientes com **perfis de comportamento distintos e coerentes** a partir dos dados observados de carteira. Esses grupos revelam que o perfil autodeclarado (conservador / moderado / arrojado) **não captura adequadamente o comportamento real** dos clientes — ou seja, declaração e prática divergem de forma sistemática.

### Por que é um problema de aprendizado não supervisionado?

Porque **não existe um gabarito de resposta correta**. O `perfil_declarado_pelo_cliente` é o que o cliente disse sobre si mesmo ao abrir a conta — não uma validação do seu comportamento real. Não há um rótulo "grupo correto" para cada cliente que o modelo deva aprender a prever.

Queremos que o modelo **descubra padrões nos dados por conta própria**, agrupando clientes semelhantes sem nenhuma supervisão humana. Isso é, por definição, clustering — uma técnica de aprendizado não supervisionado.

### Decisão analítica sobre `perfil_declarado_pelo_cliente`

Esta variável **não será usada como feature no modelo**.

O motivo é direto: incluí-la contaminaria o modelo com exatamente a informação que queremos questionar. Queremos descobrir os grupos pelo comportamento real e, *depois*, cruzar com o que foi declarado — esse cruzamento é a resposta à hipótese acima e à pergunta central do Rafael.

`id_cliente` também será excluído: é apenas um identificador único, sem valor informacional.

---
## 2. Imports e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

In [ ]:
df = pd.read_csv("carteiras_clientes.csv", encoding="ISO-8859-1")

print(f"Shape: {df.shape}")
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nValores nulos por coluna:")
print(df.isnull().sum())

---
## 3. Análise Exploratória dos Dados (EDA)

### 3.1 Estatísticas Descritivas

In [ ]:
df.describe().round(2)

**Observações imediatas:**

- **`patrimonio_declarado_R$` e `valor_total_investido_R$`**: fortemente assimétricas à direita. A média (~R\$ 597k e ~R\$ 429k) é bem acima da mediana (~R\$ 492k e ~R\$ 326k), indicando que poucos clientes com patrimônio muito alto puxam a média para cima. Normalização será obrigatória.
- **`operacoes_por_mes`**: mínimo 0, máximo 56, mediana apenas 3. Há clientes completamente inativos e outros operando quase todo dia útil no mesmo dataset.
- **`prazo_medio_posicao_dias`**: amplitude de 3 a 927 dias — perfis de day trader e buy-and-hold extremo convivem na mesma base.
- **`perc_renda_fixa`**: a mediana de 32,6% com desvio padrão de 30,9 indica uma distribuição muito espalhada — há dois grupos distintos de alocação (alta e baixa renda fixa).

### 3.2 Distribuição dos Perfis Declarados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order = ["conservador", "moderado", "arrojado"]
palette = ["#4C72B0", "#DD8452", "#55A868"]
counts = df["perfil_declarado_pelo_cliente"].value_counts()[order]

axes[0].bar(order, counts, color=palette)
axes[0].set_title("Contagem absoluta por perfil declarado")
axes[0].set_ylabel("Quantidade de clientes")
for i, v in enumerate(counts):
    axes[0].text(i, v + 2, str(v), ha="center", fontweight="bold")

axes[1].pie(counts / counts.sum() * 100, labels=order, autopct="%1.1f%%",
            colors=palette, startangle=90)
axes[1].set_title("Proporção por perfil declarado")

plt.suptitle("Perfis autodeclarados na base Vértice (n=500)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

A base é razoavelmente equilibrada: **35,6% conservador, 38,4% moderado e 26% arrojado**. Nenhum perfil é raro, o que permite comparações robustas na análise de divergência posterior.

### 3.3 Distribuições das Variáveis Numéricas

In [ ]:
num_cols = [
    "idade", "anos_como_cliente", "patrimonio_declarado_R$", "valor_total_investido_R$",
    "perc_renda_fixa", "perc_acoes", "perc_fiis", "perc_crypto", "perc_exterior",
    "qtd_ativos_distintos", "operacoes_por_mes", "prazo_medio_posicao_dias", "maior_posicao_pct_carteira"
]

fig, axes = plt.subplots(3, 5, figsize=(20, 11))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, edgecolor="white", color="#4C72B0", alpha=0.85)
    axes[i].set_title(col, fontsize=8.5)
    axes[i].set_ylabel("frequência", fontsize=7.5)
    axes[i].tick_params(labelsize=7)

for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuições das variáveis numéricas", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Padrões relevantes:**

- **`perc_renda_fixa`**: distribuição bimodal — há uma concentração próxima de 0% e outra em torno de 75–90%. Isso já sugere visualmente a existência de pelo menos dois perfis distintos de alocação.
- **`operacoes_por_mes`**: concentrada em 0–5, com cauda longa. 60 clientes não fizeram nenhuma operação; 98 fizeram mais de 10. Comportamentos opostos na mesma base.
- **`prazo_medio_posicao_dias`**: distribuição assimétrica à direita. 56 clientes com prazo < 30 dias (perfil de trader); 158 com prazo > 365 dias (buy-and-hold de longo prazo).
- **`patrimonio_declarado_R$`** e **`valor_total_investido_R$`**: muito parecidas entre si (correlação de 0.96 — confirmado adiante). Manter ambas daria peso duplo ao porte financeiro do cliente.

### 3.4 Identificação de Outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

outlier_vars = [
    ("patrimonio_declarado_R$", "Patrimônio Declarado (R$)"),
    ("operacoes_por_mes", "Operações por Mês"),
    ("prazo_medio_posicao_dias", "Prazo Médio Posição (dias)"),
    ("qtd_ativos_distintos", "Qtd. Ativos Distintos"),
]

for ax, (col, label) in zip(axes, outlier_vars):
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor="#AEC6CF"),
               medianprops=dict(color="black", linewidth=2),
               flierprops=dict(marker="o", markerfacecolor="#E74C3C", markersize=4, alpha=0.6))
    ax.set_title(label, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle("Boxplots — variáveis com maior potencial de outlier", fontsize=12)
plt.tight_layout()
plt.show()

print("Outliers detectados pelo critério IQR (fora de Q1-1.5*IQR ou Q3+1.5*IQR):\n")
for col, label in outlier_vars:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f"  {label:35s}: {n_out:3d} outliers  (Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f})")

**Impacto no K-Means:**  
O K-Means minimiza a soma das distâncias euclidianas ao centróide. Outliers puxam os centróides para longe da maioria dos pontos, podendo criar clusters artificiais para acomodar poucos casos extremos.

**Estratégia:** normalizar com `StandardScaler` (z-score) reduz o impacto da escala absoluta. Os outliers ainda existirão relativamente, mas não dominarão o cálculo de distância por conta de unidade. Na interpretação dos clusters, avaliaremos se algum grupo foi formado principalmente por extremos.

### 3.5 Mapa de Correlações

In [ ]:
num_cols_corr = [
    "idade", "anos_como_cliente", "patrimonio_declarado_R$", "valor_total_investido_R$",
    "perc_renda_fixa", "perc_acoes", "perc_fiis", "perc_crypto", "perc_exterior",
    "qtd_ativos_distintos", "operacoes_por_mes", "prazo_medio_posicao_dias", "maior_posicao_pct_carteira"
]

corr_matrix = df[num_cols_corr].corr()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True,
    cbar_kws={"shrink": 0.75},
    annot_kws={"size": 8}
)
plt.title("Matriz de correlação — variáveis numéricas", fontsize=13, pad=15)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

**Correlações mais relevantes para o modelo:**

| Par de variáveis | r | Interpretação |
|---|:---:|---|
| `patrimonio_declarado` × `valor_total_investido` | **+0.96** | Quase redundantes — manter apenas `valor_total_investido` |
| `perc_renda_fixa` × `perc_acoes` | **−0.89** | Principal eixo de diferenciação de perfil |
| `perc_crypto` × `operacoes_por_mes` | **+0.61** | Quem tem cripto opera mais ativamente |
| `qtd_ativos_distintos` × `maior_posicao_pct_carteira` | **−0.62** | Mais ativos → menor concentração (esperado) |
| `operacoes_por_mes` × `prazo_medio_posicao_dias` | **−0.51** | Trader ativo mantém posições por menos tempo |
| `perc_renda_fixa` × `operacoes_por_mes` | **−0.59** | Investidor de renda fixa opera raramente |

A correlação de **0.96** entre patrimônio declarado e valor investido é o ponto mais crítico: manter as duas no modelo daria peso duplo ao porte financeiro do cliente, distorcendo os clusters. `patrimonio_declarado_R$` será removido.

### 3.6 Divergência: Perfil Declarado vs. Comportamento Real

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
perfil_order = ["conservador", "moderado", "arrojado"]
colors = ["#4C72B0", "#DD8452", "#55A868"]

vars_analise = [
    ("perc_renda_fixa", "% Renda Fixa"),
    ("perc_acoes", "% Ações"),
    ("operacoes_por_mes", "Operações por Mês"),
]

for ax, (col, label) in zip(axes, vars_analise):
    data = [df[df["perfil_declarado_pelo_cliente"] == p][col].values for p in perfil_order]
    bp = ax.boxplot(data, patch_artist=True, labels=perfil_order)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for median in bp["medians"]:
        median.set(color="black", linewidth=2)
    ax.set_title(f"{label} por perfil declarado", fontsize=10)
    ax.set_ylabel(label, fontsize=9)

plt.suptitle("Comportamento real vs. perfil autodeclarado", fontsize=12)
plt.tight_layout()
plt.show()

# Números que sustentam a hipótese
total_cons = (df["perfil_declarado_pelo_cliente"] == "conservador").sum()
total_arr  = (df["perfil_declarado_pelo_cliente"] == "arrojado").sum()

c1 = ((df["perfil_declarado_pelo_cliente"]=="conservador") & (df["perc_renda_fixa"] < 50)).sum()
c2 = ((df["perfil_declarado_pelo_cliente"]=="conservador") & (df["perc_crypto"] > 10)).sum()
c3 = ((df["perfil_declarado_pelo_cliente"]=="conservador") & (df["perc_acoes"] > 30)).sum()
a1 = ((df["perfil_declarado_pelo_cliente"]=="arrojado")    & (df["perc_renda_fixa"] > 70)).sum()

print("=" * 55)
print("Divergências declarado × comportamento")
print("=" * 55)
print(f"Conservadores com < 50% em renda fixa : {c1}/{total_cons} ({c1/total_cons*100:.1f}%)")
print(f"Conservadores com > 10% em cripto     : {c2}/{total_cons} ({c2/total_cons*100:.1f}%)")
print(f"Conservadores com > 30% em ações      : {c3}/{total_cons} ({c3/total_cons*100:.1f}%)")
print(f"Arrojados com > 70% em renda fixa     : {a1}/{total_arr}  ({a1/total_arr*100:.1f}%)")

**A hipótese do Rafael se sustenta nos dados:**

- **15,2%** dos conservadores têm menos da metade da carteira em renda fixa.
- **4,5%** dos conservadores têm mais de 10% em cripto — ativo de altíssima volatilidade.
- **3,8%** dos arrojados têm mais de 70% em renda fixa — comportamento totalmente oposto ao perfil declarado.

Esses números indicam que o formulário de suitability não reflete o comportamento observado de forma consistente. O clustering vai quantificar quão sistemática é essa divergência.

### 3.7 Relação Entre Variáveis de Alocação

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pares = [
    ("perc_renda_fixa", "perc_acoes", "RF × Ações"),
    ("perc_acoes", "perc_crypto", "Ações × Cripto"),
    ("operacoes_por_mes", "prazo_medio_posicao_dias", "Operações × Prazo médio"),
]
palette = {"conservador": "#4C72B0", "moderado": "#DD8452", "arrojado": "#55A868"}

for ax, (x, y, title) in zip(axes, pares):
    for perfil, color in palette.items():
        subset = df[df["perfil_declarado_pelo_cliente"] == perfil]
        ax.scatter(subset[x], subset[y], c=color, alpha=0.45, s=22, label=perfil)
    ax.set_xlabel(x, fontsize=9)
    ax.set_ylabel(y, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Scatter de variáveis comportamentais (cor = perfil declarado)", fontsize=12)
plt.tight_layout()
plt.show()

No scatter **RF × Ações**, os três perfis se misturam amplamente — não existe separação linear entre eles. Isso reforça visualmente que o perfil declarado não organiza os dados de forma coerente, abrindo espaço para que o clustering encontre agrupamentos mais significativos.

No scatter **Operações × Prazo**, a anticorrelação é clara: traders ativos (muitas operações) têm prazo curto; investidores passivos (poucas operações) têm prazo longo. Essa dimensão comportamental é uma das mais discriminantes para a separação dos clusters.

---
## 4. Pré-processamento

### 4.1 Seleção de Variáveis

| Variável | Decisão | Justificativa |
|---|:---:|---|
| `id_cliente` | ❌ Remover | Identificador — sem valor informacional |
| `perfil_declarado_pelo_cliente` | ❌ Remover | É o que queremos questionar, não uma feature preditora |
| `tem_assessor` | ❌ Remover | Canal de atendimento, não descreve como o cliente investe |
| `patrimonio_declarado_R$` | ❌ Remover | Correlação de 0.96 com `valor_total_investido_R$` — redundante |
| `idade` | ✅ Usar | Contexto demográfico — horizontes e tolerâncias diferentes |
| `anos_como_cliente` | ✅ Usar | Maturidade na corretora e consolidação do comportamento |
| `valor_total_investido_R$` | ✅ Usar | Porte real da carteira (preferível ao declarado) |
| `perc_renda_fixa` | ✅ Usar | Principal eixo de diferenciação de perfil |
| `perc_acoes` | ✅ Usar | Anticorrelação com RF — dimensão de risco de mercado |
| `perc_fiis` | ✅ Usar | Estratégia de renda passiva imobiliária |
| `perc_crypto` | ✅ Usar | Alta correlação com comportamento ativo de risco |
| `perc_exterior` | ✅ Usar | Sofisticação e diversificação global |
| `qtd_ativos_distintos` | ✅ Usar | Nível de diversificação e engajamento |
| `operacoes_por_mes` | ✅ Usar | Trading ativo vs. investimento passivo |
| `prazo_medio_posicao_dias` | ✅ Usar | Day trader vs. buy-and-hold |
| `maior_posicao_pct_carteira` | ✅ Usar | Concentração de risco |

**Total: 12 features para o modelo.**

### 4.2 Normalização

**Por que normalizar?**  
O K-Means usa distância euclidiana. Sem normalização, `valor_total_investido_R$` (escala de milhões) dominaria o cálculo em relação a `operacoes_por_mes` (escala 0–56), tornando o modelo praticamente um agrupamento por tamanho de carteira.

**Método escolhido: `StandardScaler` (z-score)**  
Cada variável terá média 0 e desvio padrão 1. Preferido ao MinMaxScaler porque é menos sensível a outliers extremos — o MinMaxScaler comprimiria todos os valores não-extremos num range estreito se houver um outlier muito distante.

In [ ]:
features = [
    "idade", "anos_como_cliente", "valor_total_investido_R$",
    "perc_renda_fixa", "perc_acoes", "perc_fiis", "perc_crypto", "perc_exterior",
    "qtd_ativos_distintos", "operacoes_por_mes", "prazo_medio_posicao_dias", "maior_posicao_pct_carteira"
]

X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verificação pós-normalização
means = X_scaled.mean(axis=0).round(3)
stds  = X_scaled.std(axis=0).round(3)

pd.DataFrame({"média_após_scale": means, "std_após_scale": stds},
             index=features)

Todas as variáveis têm média ≈ 0 e desvio padrão ≈ 1 após o escalonamento. Os dados estão prontos para o K-Means.

---
## 5. Modelagem — Escolha do K

Dois critérios foram usados para determinar o número ideal de clusters:

- **Elbow Method (inércia):** mede a soma das distâncias quadradas de cada ponto ao centróide do seu cluster. Quanto menor, mais compactos os clusters — mas sempre diminui com K maior. O "cotovelo" é o ponto onde o ganho começa a ser marginal.
- **Silhouette Score:** mede quão bem cada ponto está no seu cluster comparado ao cluster vizinho mais próximo. Varia de −1 a +1; valores mais altos indicam clusters mais bem separados.

In [ ]:
inertias = []
silhouettes = []
ks = range(2, 11)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(ks), inertias, "o-", color="#4C72B0", linewidth=2, markersize=7)
axes[0].axvline(x=5, color="#E74C3C", linestyle="--", alpha=0.7, label="K=5 (escolhido)")
axes[0].set_title("Elbow Method — Inércia por K")
axes[0].set_xlabel("Número de clusters (K)")
axes[0].set_ylabel("Inércia")
axes[0].legend()
axes[0].set_xticks(list(ks))

# Silhouette
axes[1].plot(list(ks), silhouettes, "s-", color="#55A868", linewidth=2, markersize=7)
axes[1].axvline(x=5, color="#E74C3C", linestyle="--", alpha=0.7, label="K=5 (escolhido)")
axes[1].set_title("Silhouette Score por K")
axes[1].set_xlabel("Número de clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()
axes[1].set_xticks(list(ks))

plt.suptitle("Critérios para escolha do K", fontsize=13)
plt.tight_layout()
plt.show()

print(f"{'K':>3}  {'Inércia':>10}  {'Silhouette':>12}")
print("-" * 30)
for k, ine, sil in zip(ks, inertias, silhouettes):
    mark = " ◄ escolhido" if k == 5 else ""
    print(f"{k:>3}  {ine:>10.1f}  {sil:>12.4f}{mark}")

### Justificativa da escolha: K = 5

**Elbow:** A inércia cai acentuadamente de K=2 a K=5. Entre K=5 e K=6, a queda se torna muito menor, caracterizando o cotovelo. Ganhos adicionais a partir de K=6 são marginais e provavelmente fragmentam clusters coerentes em subgrupos artificiais.

**Silhouette:** O pico ocorre em K=5 (score ≈ 0.34). A partir de K=6, o score cai consistentemente, indicando que clusters adicionais não melhoram a separação entre grupos — pelo contrário, criam sobreposição.

**Interpretabilidade de negócio:** 5 perfis é um número operacional para a equipe de relacionamento da Vértice — suficiente para diferenciar abordagens sem criar complexidade de gestão. Mais de 6 clusters seria difícil de traduzir em estratégias distintas de atendimento.

---
## 6. Treinamento do Modelo Final

In [ ]:
km_final = KMeans(n_clusters=5, random_state=42, n_init=20, max_iter=500)
df["cluster"] = km_final.fit_predict(X_scaled)

sil_global = silhouette_score(X_scaled, df["cluster"])
print(f"Silhouette Score global: {sil_global:.4f}")
print(f"\nDistribuição dos clusters:")
print(df["cluster"].value_counts().sort_index().rename("contagem").to_frame()
      .assign(pct=lambda x: (x["contagem"]/len(df)*100).round(1).astype(str)+"%"))

---
## 7. Análise e Interpretação dos Clusters

### 7.1 Perfil Estatístico por Cluster

In [ ]:
summary = df.groupby("cluster")[features].mean().round(2)

# Renomear colunas para facilitar leitura
col_labels = {
    "idade": "Idade", "anos_como_cliente": "Anos cliente",
    "valor_total_investido_R$": "Investido (R$)",
    "perc_renda_fixa": "% RF", "perc_acoes": "% Ações",
    "perc_fiis": "% FIIs", "perc_crypto": "% Crypto",
    "perc_exterior": "% Exterior", "qtd_ativos_distintos": "Qtd ativos",
    "operacoes_por_mes": "Ops/mês", "prazo_medio_posicao_dias": "Prazo (dias)",
    "maior_posicao_pct_carteira": "Maior posição %"
}
summary.rename(columns=col_labels)

### 7.2 Heatmap dos Perfis (valores normalizados)

In [ ]:
# Normalizar centróides para visualização comparativa (0 a 1 por feature)
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler()
summary_norm = pd.DataFrame(
    mms.fit_transform(summary.T).T,
    columns=summary.columns,
    index=summary.index
)

nomes_clusters = {0: "Guardião Conservador", 1: "Trader Ativo", 2: "Investidor de Alto Valor",
                  3: "Diversificador Moderado", 4: "Iniciante Conservador"}

summary_norm.index = [f"C{i} — {nomes_clusters[i]}" for i in summary_norm.index]

plt.figure(figsize=(14, 5))
sns.heatmap(
    summary_norm.rename(columns=col_labels),
    annot=summary.rename(columns=col_labels).values,
    fmt=".0f",
    cmap="YlOrRd", linewidths=0.5,
    cbar_kws={"label": "Intensidade relativa (0=mín, 1=máx)"},
    annot_kws={"size": 8}
)
plt.title("Perfil médio dos clusters (cor = posição relativa entre clusters)", fontsize=12, pad=12)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

### 7.3 Distribuição das Variáveis-chave por Cluster

In [ ]:
cluster_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
cluster_labels = [f"C{i}\n{nomes_clusters[i][:12]}" for i in range(5)]

key_vars = [
    ("perc_renda_fixa",          "% Renda Fixa"),
    ("perc_acoes",               "% Ações"),
    ("operacoes_por_mes",        "Operações/mês"),
    ("prazo_medio_posicao_dias", "Prazo médio (dias)"),
    ("valor_total_investido_R$", "Valor Investido (R$)"),
    ("maior_posicao_pct_carteira","Maior posição %"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (col, label) in zip(axes, key_vars):
    data = [df[df["cluster"] == c][col].values for c in range(5)]
    bp = ax.boxplot(data, patch_artist=True, labels=cluster_labels)
    for patch, color in zip(bp["boxes"], cluster_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    for median in bp["medians"]:
        median.set(color="black", linewidth=2)
    ax.set_title(label, fontsize=10)
    ax.tick_params(axis="x", labelsize=7.5)

plt.suptitle("Distribuição das variáveis-chave por cluster", fontsize=13)
plt.tight_layout()
plt.show()

### 7.4 Visualização dos Clusters em 2D (PCA)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(11, 7))

for c, (color, nome) in enumerate(zip(cluster_colors, nomes_clusters.values())):
    mask = df["cluster"] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, alpha=0.6, s=35, label=f"C{c} — {nome}", edgecolors="white", linewidths=0.3)

# Centróides no espaço PCA
centroids_pca = pca.transform(km_final.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           c="black", marker="X", s=180, zorder=5, label="Centróides")

ax.set_xlabel(f"PC1 ({var_exp[0]*100:.1f}% da variância)", fontsize=10)
ax.set_ylabel(f"PC2 ({var_exp[1]*100:.1f}% da variância)", fontsize=10)
ax.set_title(f"Clusters em 2D via PCA (variância explicada: {sum(var_exp)*100:.1f}%)", fontsize=12)
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()

print(f"Variância explicada — PC1: {var_exp[0]*100:.1f}%  |  PC2: {var_exp[1]*100:.1f}%  |  Total: {sum(var_exp)*100:.1f}%")

O PCA captura **51,6% da variância** total nas duas componentes. É suficiente para uma visualização orientativa — clusters bem separados no gráfico indicam separação real no espaço original de 12 dimensões, mas sobreposição visual não significa necessariamente má qualidade do modelo (as dimensões não capturadas pelo PCA também contribuem para a separação).

### 7.5 Silhouette por Cluster

In [ ]:
sil_samples = silhouette_samples(X_scaled, df["cluster"])
df["silhouette"] = sil_samples

print(f"Silhouette global: {sil_global:.4f}\n")
print("Silhouette médio por cluster:")
sil_por_cluster = df.groupby("cluster")["silhouette"].mean().round(4)
for c, s in sil_por_cluster.items():
    print(f"  C{c} — {nomes_clusters[c]:25s}: {s:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
for c in range(5):
    sil_vals = np.sort(sil_samples[df["cluster"] == c])
    size_c = len(sil_vals)
    y_upper = y_lower + size_c
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_vals,
                     facecolor=cluster_colors[c], alpha=0.7)
    ax.text(-0.05, y_lower + size_c / 2, f"C{c}", fontsize=9)
    y_lower = y_upper + 10

ax.axvline(x=sil_global, color="red", linestyle="--", label=f"Score global = {sil_global:.3f}")
ax.set_xlabel("Silhouette coefficient", fontsize=10)
ax.set_ylabel("Cluster", fontsize=10)
ax.set_title("Silhouette plot — qualidade da separação por cluster", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Interpretação:**

- **C4 (Iniciante Conservador)** tem o silhouette mais alto (~0.59): o grupo mais coeso e bem separado dos demais. Faz sentido — são clientes com patrimônio muito baixo, jovens e com alocação quase toda em renda fixa, formando um cluster bem definido.
- **C1 (Trader Ativo)** e **C2 (Investidor de Alto Valor)** têm scores intermediários (~0.33 e 0.30) — razoáveis para dados de comportamento humano, que naturalmente têm sobreposição.
- **C0 e C3** têm scores menores (~0.26), indicando mais sobreposição com clusters vizinhos. São perfis de transição — o conservador estabelecido e o diversificador moderado são mais próximos entre si do que os perfis extremos.

---
## 8. Nomenclatura e Descrição dos Perfis

### Cluster 0 — Guardião Conservador
**Tamanho: 119 clientes (23,8% da base)**

**Características:**
- Idade média: **54 anos** | Anos na corretora: **6,8**
- Valor investido: **R\$ 382k** (patrimônio médio-alto)
- Alocação: **71% renda fixa**, 8% ações, 12% FIIs
- Operações por mês: **1,6** | Prazo médio: **526 dias**
- 71% tem assessor vinculado

**Quem é:** Investidor maduro, próximo ou já na fase de preservação de patrimônio. Investe com foco em segurança e renda. Opera raramente, mantém posições por muito tempo. Alta concentração em renda fixa e FIIs como fonte de renda passiva. A presença massiva de assessor (71%) indica que esses clientes já têm relacionamento estruturado.

---

### Cluster 1 — Trader Ativo
**Tamanho: 100 clientes (20% da base)**

**Características:**
- Idade média: **32 anos** | Anos na corretora: **3,1**
- Valor investido: **R\$ 271k**
- Alocação: **52% ações**, 22% cripto, 8% RF
- Operações por mês: **22** | Prazo médio: **30 dias**
- 74% NÃO tem assessor

**Quem é:** Jovem, engajado e com perfil especulativo. Opera com altíssima frequência e mantém posições por pouco tempo. A combinação de alta exposição em ações e cripto revela tolerância ao risco elevada. A ausência de assessor (74%) sugere investidor autônomo, que busca informações por conta própria e toma decisões sem intermediação.

---

### Cluster 2 — Investidor de Alto Valor
**Tamanho: 80 clientes (16% da base)**

**Características:**
- Idade média: **48 anos** | Anos na corretora: **8,5**
- Valor investido: **R\$ 1.017k** (maior patrimônio da base)
- Alocação: **64% ações**, 14% cripto, 10% RF
- Operações por mês: **6** | Prazo médio: **98 dias**
- Maior posição: **68%** da carteira

**Quem é:** Investidor experiente, de alto patrimônio e com carteira concentrada em renda variável. Não é trader — opera com moderação e horizonte de médio prazo. A concentração altíssima (maior posição = 68% da carteira) indica convicção em poucos ativos de qualidade. É o cliente mais rentável potencial da corretora.

---

### Cluster 3 — Diversificador Moderado
**Tamanho: 110 clientes (22% da base)**

**Características:**
- Idade média: **41 anos** | Anos na corretora: **5,3**
- Valor investido: **R\$ 523k**
- Alocação: **29% RF**, 32% ações, 28% FIIs, 8% exterior
- Operações por mês: **3,5** | Prazo médio: **391 dias**
- 56% tem assessor

**Quem é:** Perfil equilibrado em todos os sentidos. Distribui a carteira entre renda fixa, ações e FIIs de forma relativamente uniforme, com alguma exposição internacional. Opera pouco e mantém posições por longo prazo. É o perfil mais "manual" de descrever — não tem característica dominante, mas é justamente isso que o define: diversificação consciente.

---

### Cluster 4 — Iniciante Conservador
**Tamanho: 91 clientes (18,2% da base)**

**Características:**
- Idade média: **30 anos** | Anos na corretora: **1,2**
- Valor investido: **R\$ 34k** (menor patrimônio da base)
- Alocação: **80% renda fixa**, 9% ações, 5% FIIs
- Operações por mês: **0,7** | Prazo médio: **173 dias**
- 74% NÃO tem assessor

**Quem é:** Cliente novo, jovem, com pouco capital investido e comportamento extremamente cauteloso. Coloca quase tudo em renda fixa, opera muito raramente e não tem assessor. Pode representar alguém que ainda está formando reserva de emergência ou que tem medo de tomar risco sem orientação. É o cliente com maior potencial de migração para outros perfis com a abordagem certa.

---
## 9. Hipótese do Rafael: Declaração vs. Comportamento Real

### 9.1 Cruzamento: Clusters × Perfil Declarado

In [ ]:
ct = pd.crosstab(df["cluster"], df["perfil_declarado_pelo_cliente"])
ct.index = [f"C{i} — {nomes_clusters[i]}" for i in ct.index]
print("Contagem absoluta:")
print(ct.to_string())

pct = pd.crosstab(df["cluster"], df["perfil_declarado_pelo_cliente"], normalize="index") * 100
pct.index = [f"C{i} — {nomes_clusters[i]}" for i in pct.index]
print("\nProporção dentro do cluster (%):")
print(pct.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ct_plot = pd.crosstab(df["cluster"], df["perfil_declarado_pelo_cliente"])
ct_plot.index = [f"C{i}\n{nomes_clusters[i][:14]}" for i in ct_plot.index]
ct_plot.plot(kind="bar", ax=axes[0], color=["#55A868","#4C72B0","#DD8452"], edgecolor="white", width=0.7)
axes[0].set_title("Contagem absoluta: perfil declarado por cluster", fontsize=10)
axes[0].set_xlabel("")
axes[0].set_ylabel("Quantidade de clientes")
axes[0].tick_params(axis="x", rotation=15)
axes[0].legend(title="Perfil declarado", fontsize=8)

pct_plot = pd.crosstab(df["cluster"], df["perfil_declarado_pelo_cliente"], normalize="index") * 100
pct_plot.index = [f"C{i}\n{nomes_clusters[i][:14]}" for i in pct_plot.index]
pct_plot.plot(kind="bar", stacked=True, ax=axes[1],
              color=["#55A868","#4C72B0","#DD8452"], edgecolor="white", width=0.7)
axes[1].set_title("Proporção do perfil declarado dentro de cada cluster (%)", fontsize=10)
axes[1].set_xlabel("")
axes[1].set_ylabel("% dentro do cluster")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(title="Perfil declarado", fontsize=8)

plt.suptitle("Clusters comportamentais × Perfil autodeclarado", fontsize=12)
plt.tight_layout()
plt.show()

### 9.2 O que os dados revelam sobre a hipótese do Rafael

**A hipótese se confirma — e de forma bem clara:**

| Cluster | Comportamento real | Perfil declarado dominante | Divergência? |
|---|---|---|:---:|
| C0 — Guardião Conservador | RF=71%, ops=1.6/mês | 73% conservador | ✅ Alinhado |
| C1 — Trader Ativo | Ações+Cripto=74%, ops=22/mês | **65% arrojado — mas 33% moderado** | ⚠️ Parcial |
| C2 — Investidor de Alto Valor | Ações=64%, patrimônio alto | **51% arrojado, 40% moderado** | ⚠️ Parcial |
| C3 — Diversificador Moderado | Carteira balanceada | 66% moderado | ✅ Relativamente alinhado |
| C4 — Iniciante Conservador | RF=80%, ops=0.7/mês | 70% conservador | ✅ Alinhado |

**Os casos mais críticos:**

- **C1 (Trader Ativo)**: 33% desses clientes se declararam moderados, mas operam em média 22 vezes por mês com 74% da carteira em ativos de altíssimo risco. Comportamento claramente arrojado.
- **C2 (Investidor de Alto Valor)**: 40% se declararam moderados, mas têm a carteira mais arrojada da base (64% em ações, carteira concentrada em 68%). São clientes que subestimaram o próprio perfil de risco.
- **C0 (Guardião Conservador)**: 25% se declararam moderados, mas o comportamento observado (71% em RF, opera 1.6x/mês, prazo de 526 dias) é puramente conservador.

**Conclusão:** O suitability atual da Vértice classifica corretamente os perfis extremos (conservadores muito conservadores e arrojados muito ativos), mas **falha nos casos intermediários**. Clientes moderados na declaração estão espalhados por todos os 5 clusters comportamentais, indicando que "moderado" está sendo usado como resposta padrão por clientes que não refletiram profundamente sobre o próprio perfil.

---
## 10. Resposta ao E-mail do Rafael

---

**De:** Equipe de Dados e Analytics  
**Para:** Rafael Monteiro — Head de Relacionamento com Clientes | Vértice Investimentos  
**Assunto:** Re: Precisamos de inteligência sobre nossa base de clientes

---

Rafael,

Analisamos as 500 carteiras ativas da Vértice e encontramos **cinco perfis comportamentais distintos**. A boa notícia é que eles são bem definidos e acionáveis — o time de relacionamento consegue operar a partir deles sem precisar de nenhuma adaptação técnica.

---

**Os cinco perfis:**

**1. Guardião Conservador** — 119 clientes (23,8%)  
Investidores maduros, em torno dos 54 anos, com patrimônio médio de R\$ 382k. Aplicam 71% da carteira em renda fixa e FIIs, operam raramente e mantêm posições por mais de um ano e meio. São clientes estabilizados, com baixíssima rotatividade. A maioria já tem assessor vinculado — o foco aqui é retenção e aprofundamento do relacionamento.

**2. Trader Ativo** — 100 clientes (20%)  
Jovens em torno dos 32 anos, com carteira focada em ações e cripto (74% combinados). Executam em média 22 operações por mês e raramente ficam numa posição por mais de 30 dias. 74% não tem assessor — são autônomos e bem informados. Precisam de conteúdo técnico avançado e ferramentas de análise, não de orientação básica.

**3. Investidor de Alto Valor** — 80 clientes (16%)  
O grupo mais valioso da base: patrimônio médio de R\$ 1 milhão, carteira concentrada em ações (64%). Opera com moderação — não é trader, mas assume riscos altos de forma deliberada. Esses clientes têm perfil sofisticado e merecem atenção personalizada e acesso a produtos de maior complexidade (offshore, estruturados, fundos exclusivos).

**4. Diversificador Moderado** — 110 clientes (22%)  
Perfil equilibrado: distribuem a carteira entre renda fixa, ações, FIIs e ativos internacionais de forma relativamente uniforme. Operam pouco e têm visão de longo prazo. São o grupo mais receptivo a rebalanceamentos periódicos e educação sobre diversificação global.

**5. Iniciante Conservador** — 91 clientes (18,2%)  
Clientes novos (média de 1,2 ano na corretora), jovens (~30 anos) e com pouco capital investido (média de R\$ 34k). Quase tudo está em renda fixa e operam muito raramente. São o grupo com maior potencial de crescimento — precisam de educação financeira, acompanhamento próximo e incentivos graduais para diversificar.

---

**Sobre a sua hipótese — declaração vs. comportamento:**

Você estava certo. Encontramos divergências relevantes.

Os casos mais críticos são os clientes que se declararam **moderados**: eles estão espalhados pelos cinco perfis comportamentais. Há "moderados" que operam 22 vezes por mês com 74% da carteira em ações e cripto (comportamento claramente arrojado) e há "moderados" que têm 71% em renda fixa e nunca operaram (comportamento claramente conservador).

O suitability atual acerta bem nos extremos — os muito conservadores e os muito arrojados tendem a declarar corretamente. O problema está no meio: "moderado" virou uma resposta padrão para quem não refletiu profundamente sobre o próprio perfil de risco.

**Recomendação prática:** considerar uma revisão periódica do suitability com base nos dados de comportamento real — não apenas no que o cliente declarou na abertura de conta.

---

Ficamos à disposição para aprofundar qualquer perfil ou cruzar com outros dados que vocês tenham.

Equipe de Dados e Analytics

---